# Cell-proportion clock development

Paper: Fig. 2a–c. Inputs: the original annotated healthy reference (`inte_final_healthy.h5ad`), supplied separately, and Arial fonts. Environment: Python/Scanpy, NumPy, pandas, SciPy, scikit-learn, seaborn, Matplotlib and joblib. Outputs: model-comparison CSV, Extra Trees estimator and feature metadata, prediction and feature-importance figures, all below `OUTPUT_DIR/fig2`. No dependence on another tutorial's memory. External cohort validation and Fig. 2e are not included here.

Original source indices below are **zero-based**. Original cell hashes are retained in cell metadata. Outputs were cleared; inclusion does not imply successful execution.

**Resource identity:** the author confirms this healthy PBMC dataset is the formal modeling input. Counts describe this processing stage and are not grounds to reject it based on the historical 209-donor description. The released proportion model (886 trees) equals the saved model loaded by original Fig. 2. Observation-metadata predictions agree with archived metrics; model search has not been rerun.


## Required resources and configuration

Healthy PBMC modeling data `inte_final_healthy.h5ad`, author-confirmed for this study; [healthy dataset](../RESOURCE_DOWNLOAD.md#3-healthy-pbmc-modeling-dataset).

Also supply a licensed local Arial font directory (`PERICLOCK_FONT_DIR`); see [other references](../RESOURCE_DOWNLOAD.md#8-other-reference-databases).

Download locations, sizes, SHA256, versions, permissions and publication status are maintained once in [RESOURCE_DOWNLOAD.md](../RESOURCE_DOWNLOAD.md). Individual GEO/GSM sources and sample choices are in the paper’s Supplementary Table 1. Clock models are uploaded to [Hugging Face Models](https://huggingface.co/LeyiZhang1/PeriClock); [Dataset uploads](https://huggingface.co/datasets/LeyiZhang1/PeriClock-data) are in progress. Both repositories are currently private. Consult the resource guide for per-file availability; donor-level public redistribution remains subject to review.

Set `PERICLOCK_RESOURCE_DIR` for external data and `PERICLOCK_MODEL_DIR` for models, or override individual file paths in the next cell. Keep outputs separate from inputs. Start a fresh kernel and run in order.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from within this GitHub checkout.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "periclock").is_dir() and (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from within the PeriClock code checkout")
RESOURCE_DIR = Path(os.environ.get("PERICLOCK_RESOURCE_DIR", "~/periclock-resources")).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PERICLOCK_MODEL_DIR", str(RESOURCE_DIR / "models"))).expanduser().resolve()
# Each individual file-path environment variable still overrides these defaults.
REFERENCE_H5AD = Path(os.environ.get("PERICLOCK_REFERENCE_H5AD", str(RESOURCE_DIR / "datasets/inte_final_healthy.h5ad"))).expanduser().resolve()
FONT_DIR = Path(os.environ.get("PERICLOCK_FONT_DIR", "/path/to/Arial")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("PERICLOCK_TUTORIAL_OUTPUT", str(PROJECT_ROOT / "outputs/01_proportion_clock"))).expanduser().resolve()
# Input files remain read-only; all relative output paths below use this directory.
for input_path in [REFERENCE_H5AD]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Set the input path in this cell: {input_path}")
if not FONT_DIR.is_dir():
    raise FileNotFoundError("Set PERICLOCK_FONT_DIR to a directory containing licensed Arial fonts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_DIR)

## Proportion model comparison and Extra Trees evaluation

Source: `final/fig1_2.ipynb`, cells 50, 52–53, 55–57, 59, 65–68. The 11-model search, Extra Trees evaluation and feature importance correspond to Fig. 2a–c. Historical Gradient Boosting branches are outside this tutorial. This is an expensive training workflow: it has **not** been rerun during preparation.

The reference AnnData must contain `Individual`, numeric `Age` and `cell_subtype_final` in `.obs`. Subtype labels must match the research annotation. The source uses `list(set(...))` for feature order; exact retraining repeatability across Python hash seeds is unverified. The original `< 28` missing-subtype filter is retained.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns

In [ ]:
adata_h = sc.read_h5ad(str(REFERENCE_H5AD))

In [ ]:
target_cell_types = list(set(adata_h.obs['cell_subtype_final']))

In [ ]:
### 不同模型的性能
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import ElasticNet, Ridge, Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor

from scipy.stats import randint, uniform, loguniform


def run_common_ml_models(
    adata,
    target_cell_types,
    test_size=0.2,
    random_state=42,
    max_zeros=28
):
    # =========================
    # 1. 构建细胞比例特征
    # =========================
    cell_counts = (
        adata.obs
        .groupby(["Individual", "Age", "cell_subtype_final"])
        .size()
        .unstack(fill_value=0)
    )

    cell_props = (
        cell_counts
        .div(cell_counts.sum(axis=1), axis=0)
        .dropna(axis=0, how="any")
    )

    df = cell_props.reset_index()

    df["Individual"] = df["Individual"].astype(str)
    df["Individual"] = (
        df.groupby("Individual")
        .cumcount()
        .astype(str)
        .radd(df["Individual"] + "_")
    )

    df.set_index("Individual", inplace=True)

    # =========================
    # 2. 过滤细胞类型太少的个体
    # =========================
    cell_prop_data = df.iloc[:, 2:]
    zero_counts = (cell_prop_data == 0).sum(axis=1)
    df = df[zero_counts < max_zeros].copy()

    # =========================
    # 3. 补齐目标细胞类型
    # =========================
    for ct in target_cell_types:
        if ct not in df.columns:
            df[ct] = 0

    X = df[target_cell_types].copy()
    y = df["Age"].copy()

    # =========================
    # 4. train / valid 划分
    # =========================
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state
    )

    # =========================
    # 5. 定义模型
    # =========================
    max_pls = min(10, X_train.shape[0] - 1, X_train.shape[1])
    max_pca = min(20, X_train.shape[0] - 1, X_train.shape[1])

    base_models = {
        "Linear Regression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
        "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
        "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=10000))]),
        "ElasticNet": Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(max_iter=10000, random_state=random_state))]),
        "PLSR": Pipeline([("scaler", StandardScaler()), ("model", PLSRegression())]),
        "SVR": Pipeline([("scaler", StandardScaler()), ("model", SVR())]),
        "KNN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsRegressor())]),
        "Random Forest": RandomForestRegressor(random_state=random_state, n_jobs=-1),
        "Extra Trees": ExtraTreesRegressor(random_state=random_state, n_jobs=-1),
        # "Gradient Boosting": GradientBoostingRegressor(random_state=random_state),
        "PCA + Ridge": Pipeline([("scaler", StandardScaler()), ("pca", PCA()), ("model", Ridge())]),
        "MLP": Pipeline([("scaler", StandardScaler()), ("model", MLPRegressor(max_iter=3000, random_state=random_state))])
    }

    param_spaces = {
        "Linear Regression": {},

        "Ridge": {"model__alpha": loguniform(1e-3, 100)},

        "Lasso": {"model__alpha": loguniform(1e-4, 1)},

        "ElasticNet": {
            "model__alpha": loguniform(1e-4, 1),
            "model__l1_ratio": uniform(0.05, 0.95)
        },

        "PLSR": {
            "model__n_components": randint(2, max_pls + 1)
        },

        "SVR": {
            "model__kernel": ["rbf", "linear", "poly"],
            "model__C": loguniform(1e-2, 100),
            "model__gamma": ["scale", "auto"]
        },

        "KNN": {
            "model__n_neighbors": randint(3, 20),
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2]
        },

        "Random Forest": {
            "n_estimators": randint(200, 1200),
            "max_depth": [None, 5, 10, 20],
            "min_samples_split": randint(2, 10),
            "min_samples_leaf": randint(1, 5),
            "max_features": ["sqrt", "log2", None]
        },

        "Extra Trees": {
            "n_estimators": randint(200, 1200),
            "max_depth": [None, 5, 10, 20],
            "min_samples_split": randint(2, 10),
            "min_samples_leaf": randint(1, 5),
            "max_features": ["sqrt", "log2", None]
        },

        # "Gradient Boosting": {
        #     "n_estimators": randint(100, 1000),
        #     "learning_rate": loguniform(0.005, 0.3),
        #     "max_depth": randint(2, 6),
        #     "min_samples_split": randint(2, 10),
        #     "subsample": uniform(0.6, 0.4)
        # },

        "PCA + Ridge": {
            "pca__n_components": randint(2, max_pca + 1),
            "model__alpha": loguniform(1e-3, 100)
        },

        "MLP": {
            "model__hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64), (128, 64, 32)],
            "model__activation": ["relu", "tanh"],
            "model__alpha": loguniform(1e-5, 1e-1),
            "model__learning_rate_init": loguniform(1e-4, 1e-2)
        }
    }

    # =========================
    # 6. 训练、预测、评估
    # =========================
    metrics_records = []
    pred_records = []
    fitted_models = {}
    best_params = {}
    best_cv_score = {}

    for model_name, model in base_models.items():

        print(f"\n{'=' * 60}")
        print(f"Searching {model_name}")
        print('=' * 60)

        if param_spaces[model_name]:
            search = RandomizedSearchCV(
                estimator=model,
                param_distributions=param_spaces[model_name],
                n_iter=50,
                cv=5,
                scoring="r2",
                n_jobs=-1,
                random_state=random_state,
                refit=True,
                return_train_score=True
            )
            search.fit(X_train, y_train)
            best_model = search.best_estimator_
            best_params[model_name] = search.best_params_
            best_cv_score[model_name] = search.best_score_

            print("Best CV R²:", round(search.best_score_, 3))
            print("Best Params:", search.best_params_)

        else:
            best_model = model.fit(X_train, y_train)
            best_params[model_name] = {}
            best_cv_score[model_name] = np.nan

        fitted_models[model_name] = best_model

        y_train_pred = np.ravel(best_model.predict(X_train))
        y_valid_pred = np.ravel(best_model.predict(X_valid))

        for dataset, yy, pp, idx in [
            ("train", y_train, y_train_pred, X_train.index),
            ("valid", y_valid, y_valid_pred, X_valid.index)
        ]:

            metrics_records.append({
                "model": model_name,
                "dataset": dataset,
                "R2": r2_score(yy, pp),
                "MAE": mean_absolute_error(yy, pp),
                "RMSE": np.sqrt(mean_squared_error(yy, pp))
            })

            pred_records.append(pd.DataFrame({
                "Individual": idx,
                "dataset": dataset,
                "model": model_name,
                "Age": yy.values,
                "Predicted_Age": pp
            }))
            
    metrics_df = pd.DataFrame(metrics_records)

    pred_all_df = pd.concat(pred_records, axis=0).reset_index(drop=True)

    metrics_wide = metrics_df.pivot(
        index="model",
        columns="dataset",
        values=["R2", "MAE", "RMSE"]
    )
    return {
        "metrics_df": metrics_df,
        "metrics_wide": metrics_wide,
        "pred_all_df": pred_all_df,
        "models": fitted_models,
        "best_params": best_params,
        "best_cv_score": best_cv_score,
        "X_train": X_train,
        "X_valid": X_valid,
        "y_train": y_train,
        "y_valid": y_valid
    }

In [ ]:
results = run_common_ml_models(
    adata=adata_h,
    target_cell_types=target_cell_types,
    test_size=0.2,
    random_state=42,
    max_zeros=28
)

metrics_df = results["metrics_df"]
metrics_wide = results["metrics_wide"]
pred_all_df = results["pred_all_df"]
models = results["models"]

metrics_df

In [ ]:
Path("fig2/model_comparison").mkdir(parents=True, exist_ok=True)

In [ ]:
metrics_df.to_csv('fig2/model_comparison/metrics_df.csv', index=False)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns

# =========================
# 0. Arial 字体
# =========================
font_dir = str(FONT_DIR)
for font_file in os.listdir(font_dir):
    if font_file.lower().endswith((".ttf", ".otf")):
        fm.fontManager.addfont(os.path.join(font_dir, font_file))

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.sans-serif"] = ["Arial"]
mpl.rcParams["font.weight"] = "bold"
mpl.rcParams["axes.titleweight"] = "bold"
mpl.rcParams["axes.labelweight"] = "bold"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

sns.set_style("white")

# =========================
# 1. 参数
# =========================
highlight = "#E69F00"
normal = "#8FB9DD"

title_size = 26
label_size = 22
tick_size = 18
model_size = 18
value_size = 17

# =========================
# 2. 数据
# =========================
r2_df = metrics_df[metrics_df["dataset"] == "valid"].copy()
mae_df = metrics_df[metrics_df["dataset"] == "valid"].copy()

model_order = r2_df.sort_values("R2", ascending=False)["model"].tolist()
r2_df["model"] = pd.Categorical(r2_df["model"], categories=model_order, ordered=True)
mae_df["model"] = pd.Categorical(mae_df["model"], categories=model_order, ordered=True)

r2_df = r2_df.sort_values("model")
mae_df = mae_df.sort_values("model")

r2_colors = [highlight if m == "Extra Trees" else normal for m in r2_df["model"]]
mae_colors = [highlight if m == "Extra Trees" else normal for m in mae_df["model"]]

# =========================
# 3. 作图
# =========================
fig, axes = plt.subplots(1, 2, figsize=(9.2, 6.2), sharey=True)

sns.barplot(data=r2_df, x="R2", y="model", palette=r2_colors, ax=axes[0])
axes[0].set_title("Validation R²", fontsize=title_size, fontweight="bold", pad=12)
axes[0].set_xlabel("R²", fontsize=label_size, fontweight="bold")
axes[0].set_ylabel("")

r2_min, r2_max = r2_df["R2"].min(), r2_df["R2"].max()
r2_range = r2_max - r2_min
axes[0].set_xlim(r2_min - r2_range * 0.12, r2_max + r2_range * 0.20)
axes[0].axvline(0, color="black", linewidth=1.2)

offset = r2_range * 0.015
for i, v in enumerate(r2_df["R2"]):
    axes[0].text(v + offset, i, f"{v:.2f}", va="center", ha="left", fontsize=value_size, fontfamily="Arial", fontweight="bold")

sns.barplot(data=mae_df, x="MAE", y="model", palette=mae_colors, ax=axes[1])
axes[1].set_title("Validation MAE", fontsize=title_size, fontweight="bold", pad=12)
axes[1].set_xlabel("Years", fontsize=label_size, fontweight="bold")
axes[1].set_ylabel("")

mae_max = mae_df["MAE"].max()
axes[1].set_xlim(0, mae_max * 1.15)
for i, v in enumerate(mae_df["MAE"]):
    axes[1].text(v + mae_max * 0.015, i, f"{v:.2f}", va="center", ha="left", fontsize=value_size, fontfamily="Arial", fontweight="bold")

for ax in axes:
    ax.tick_params(axis="x", labelsize=tick_size, width=1.2, length=4)
    ax.tick_params(axis="y", labelsize=model_size, width=1.2, length=0)
    for label in ax.get_xticklabels():
        label.set_fontfamily("Arial")
        label.set_fontweight("bold")
    for label in ax.get_yticklabels():
        label.set_fontfamily("Arial")
        label.set_fontweight("bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)

plt.subplots_adjust(wspace=0.14)
plt.tight_layout()

os.makedirs("fig2", exist_ok=True)
plt.savefig("fig2/Model_comparison_no_GB.pdf", dpi=300, bbox_inches="tight")
plt.savefig("fig2/Model_comparison_no_GB.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ===============================
# 0. Arial 字体设置
# ===============================
font_dir = str(FONT_DIR)
for font_file in os.listdir(font_dir):
    if font_file.lower().endswith((".ttf", ".otf")):
        fm.fontManager.addfont(os.path.join(font_dir, font_file))

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.sans-serif"] = ["Arial"]
mpl.rcParams["font.weight"] = "bold"
mpl.rcParams["axes.titleweight"] = "bold"
mpl.rcParams["axes.labelweight"] = "bold"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["axes.linewidth"] = 1.5
mpl.rcParams["xtick.major.width"] = 1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.direction"] = "out"
mpl.rcParams["ytick.direction"] = "out"

# ===============================
# 1. Extra Trees
# ===============================
et = results["models"]["Extra Trees"]
X_train, X_valid = results["X_train"], results["X_valid"]
y_train, y_valid = results["y_train"], results["y_valid"]

train_pred = et.predict(X_train)
valid_pred = et.predict(X_valid)

# ===============================
# 2. 字号和点大小
# ===============================
label_size = 24
tick_size = 20
title_size = 24
metric_size = 19
point_size = 100

# ===============================
# 3. 创建画布
# ===============================
fig, axes = plt.subplots(1, 2, figsize=(8, 3.8), dpi=600, sharex=True, sharey=True)

# ===============================
# 4. 统一坐标范围
# ===============================
xmin = min(y_train.min(), y_valid.min(), train_pred.min(), valid_pred.min())
xmax = max(y_train.max(), y_valid.max(), train_pred.max(), valid_pred.max())
padding = 2
xmin -= padding
xmax += padding

# ===============================
# 5. 数据
# ===============================
datasets = [
    (axes[0], y_train, train_pred, "Train", "#3F6FAE"),
    (axes[1], y_valid, valid_pred, "Internal test", "#D66A3A")
]

# ===============================
# 6. 作图
# ===============================
for ax, y_true, y_pred, title, color in datasets:
    ax.scatter(y_true, y_pred, s=point_size, color=color, edgecolor="none", alpha=1.0)
    ax.plot([xmin, xmax], [xmin, xmax], ls="--", lw=1.8, color="gray")

    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    ax.text(0.96, 0.05, f"R² = {r2:.3f}\nMAE = {mae:.2f}\nRMSE = {rmse:.2f}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=metric_size,
            fontfamily="Arial", fontweight="bold", linespacing=1.15,
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="lightgray", linewidth=1.0))

    ax.text(0.04, 0.95, title, transform=ax.transAxes, ha="left", va="top",
            fontsize=title_size, fontfamily="Arial", fontweight="bold")

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(xmin, xmax)
    ax.set_xlabel("Chronological age", fontsize=label_size, fontfamily="Arial", fontweight="bold", labelpad=6)
    ax.tick_params(axis="both", labelsize=tick_size, width=1.5, length=5)

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontfamily("Arial")
        label.set_fontweight("bold")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["bottom"].set_linewidth(1.5)

# ===============================
# 7. Y轴标题
# ===============================
axes[0].set_ylabel("Predicted age", fontsize=label_size, fontfamily="Arial", fontweight="bold", labelpad=6)

# ===============================
# 8. 布局
# ===============================
plt.tight_layout(w_pad=1.5)

# ===============================
# 9. 保存
# ===============================
os.makedirs("fig2", exist_ok=True)
plt.savefig("fig2/ExtraTrees.pdf", dpi=300, bbox_inches="tight")
plt.savefig("fig2/ExtraTrees.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import os
import joblib
import pandas as pd
import json

# =========================
# 保存目录
# =========================
save_dir = "fig2/ET_model"
os.makedirs(save_dir, exist_ok=True)

# =========================
# 1. 获取模型
# =========================
et = results["models"]["Extra Trees"]

# =========================
# 2. 获取模型实际使用的特征
# =========================
X_train = results["X_train"]

# 如果 X_train 是 pandas DataFrame
if hasattr(X_train, "columns"):
    feature_names = X_train.columns.tolist()
else:
    raise ValueError(
        "X_train 不是 DataFrame，无法自动获取特征名称。请使用训练模型时的原始特征列表。"
    )

print("Number of features:", len(feature_names))
print(feature_names[:10])

# =========================
# 3. 保存模型
# =========================
joblib.dump(
    et,
    os.path.join(save_dir, "ExtraTrees_model.joblib")
)

# =========================
# 4. 保存特征列表
# =========================
pd.DataFrame({
    "feature": feature_names
}).to_csv(
    os.path.join(save_dir, "ExtraTrees_features.csv"),
    index=False
)

# 同时保存 json，后续读取更方便
with open(
    os.path.join(save_dir, "ExtraTrees_features.json"),
    "w"
) as f:
    json.dump(feature_names, f, indent=2)

# =========================
# 5. 保存模型参数
# =========================
model_params = et.get_params()

with open(
    os.path.join(save_dir, "ExtraTrees_model_params.json"),
    "w"
) as f:
    json.dump(model_params, f, indent=2, default=str)

print("Saved successfully:")
print(os.path.join(save_dir, "ExtraTrees_model.joblib"))
print(os.path.join(save_dir, "ExtraTrees_features.csv"))
print(os.path.join(save_dir, "ExtraTrees_features.json"))
print(os.path.join(save_dir, "ExtraTrees_model_params.json"))

In [ ]:
import pandas as pd

# Extra Trees模型
et = results["models"]["Extra Trees"]

# 特征重要性
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": et.feature_importances_
})

importance_df = (
    importance_df
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

importance_df.head(10)

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns

# ==========================
# 0. Arial 字体设置
# ==========================
font_dir = str(FONT_DIR)

for font_file in os.listdir(font_dir):
    if font_file.lower().endswith((".ttf", ".otf")):
        fm.fontManager.addfont(os.path.join(font_dir, font_file))

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.sans-serif"] = ["Arial"]
mpl.rcParams["font.weight"] = "bold"
mpl.rcParams["axes.titleweight"] = "bold"
mpl.rcParams["axes.labelweight"] = "bold"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["axes.linewidth"] = 1.5
mpl.rcParams["xtick.major.width"] = 1.5
mpl.rcParams["ytick.major.width"] = 1.5

# ==========================
# 1. 取 importance 前10
# ==========================
top10 = (
    importance_df
    .sort_values("importance", ascending=False)
    .head(10)
    .copy()
)

# ==========================
# 2. 论文风格
# ==========================
sns.set_style("white")

fig, ax = plt.subplots(
    figsize=(5.4, 4.2),
    dpi=600
)

# 深蓝 → 浅蓝渐变
colors = sns.color_palette("Blues_r", len(top10))

sns.barplot(
    data=top10,
    x="importance",
    y="feature",
    palette=colors,
    ax=ax
)

# ==========================
# 3. 坐标轴
# ==========================
ax.set_xlabel(
    "Feature Importance",
    fontsize=17,
    fontfamily="Arial",
    fontweight="bold",
    labelpad=6
)

ax.set_ylabel("")

ax.set_title(
    "Top 10 Important Features",
    fontsize=18,
    fontfamily="Arial",
    fontweight="bold",
    pad=10
)

# X轴刻度
ax.tick_params(
    axis="x",
    labelsize=15,
    width=1.5,
    length=4
)

# Y轴特征名称
ax.tick_params(
    axis="y",
    labelsize=15,
    width=1.5,
    length=0
)

# 强制所有刻度 Arial + Bold
for label in ax.get_xticklabels():
    label.set_fontfamily("Arial")
    label.set_fontweight("bold")

for label in ax.get_yticklabels():
    label.set_fontfamily("Arial")
    label.set_fontweight("bold")

# ==========================
# 4. 边框
# ==========================
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.5)
ax.spines["bottom"].set_linewidth(1.5)

# ==========================
# 5. 数值标注
# ==========================
max_importance = top10["importance"].max()

for i, value in enumerate(top10["importance"]):
    ax.text(
        value + max_importance * 0.015,
        i,
        f"{value:.2f}",
        va="center",
        ha="left",
        fontsize=14,
        fontfamily="Arial",
        fontweight="bold"
    )

# 给右侧数值留出空间
ax.set_xlim(
    0,
    max_importance * 1.16
)

# ==========================
# 6. 保存
# ==========================
plt.tight_layout()

os.makedirs("fig2", exist_ok=True)

plt.savefig(
    "fig2/ET_Top10_Importance.pdf",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "fig2/ET_Top10_Importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()